In [ ]:
import tkinter as tk
from tkinter import messagebox
import threading
from translation import translate_text
from text_to_speech import text_to_speech
from speech_to_text_translation import handle_speech
from googletrans import Translator

# Dictionary of languages and their codes
LANGUAGES = {
    'en': 'English',
    'es': 'Spanish',
    'fr': 'French',
    'de': 'German',
    'it': 'Italian',
    'pt': 'Portuguese',
    'ru': 'Russian',
    'ar': 'Arabic',
    'zh': 'Chinese',
    'ja': 'Japanese',
    'hi': 'Hindi',
    'ko': 'Korean',
    'tr': 'Turkish',
}

def detect_language(input_text):
    translator = Translator()
    detected_lang = translator.detect(input_text).lang
    return detected_lang

def submit_text():
    text_input = text_entry.get("1.0", tk.END).strip()
    preferred_language = preferred_language_var.get()

    if text_input:
        detected_language = detect_language(text_input)
        print(f"Detected language: {detected_language}")
        translated_text = translate_text(text_input, preferred_language)

        # Show translated output
        text_to_speech(translated_text, preferred_language)
        output_text.delete("1.0", tk.END)
        output_text.insert(tk.END, f"Translated to {LANGUAGES.get(preferred_language, preferred_language)}: \n{translated_text}")

        detected_language_label.config(text=f"Detected Language: {LANGUAGES.get(detected_language, detected_language)}")
        preferred_language_label.config(text=f"Preferred Language: {LANGUAGES.get(preferred_language, preferred_language)}")
    else:
        messagebox.showwarning("Input Required", "Please enter text to detect language.")

def convert_to_speech():
    text_input = text_entry.get("1.0", tk.END).strip()
    preferred_language = preferred_language_var.get()

    if text_input:
        detected_language = detect_language(text_input)
        print(f"Detected Language for Speech-to-Text: {detected_language}")
        translated_text = translate_text(text_input, detected_language)
        text_to_speech(translated_text, detected_language)
    else:
        messagebox.showwarning("Input Required", "Please enter text to convert to speech.")

def handle_speech_gui():
    def background_speech_recognition():
        preferred_language = preferred_language_var.get()
        status_label.config(text="Listening...", fg="orange")
        root.update()
        recognized_text = handle_speech('')
        print(f"Recognized text from speech: {recognized_text}")
        
        if recognized_text:
            detected_language = detect_language(recognized_text)
            print(f"Detected language from speech: {detected_language}")

            translated_text = translate_text(recognized_text, preferred_language)
            
            def update_gui_with_speech():
                output_text.delete("1.0", tk.END)
                output_text.insert(tk.END, f"Recognized and Translated to {LANGUAGES.get(preferred_language, preferred_language)}: \n{translated_text}")
                text_to_speech(translated_text, preferred_language)
                detected_language_label.config(text=f"Detected Language: {LANGUAGES.get(detected_language, detected_language)}")
                status_label.config(text="Ready", fg="green")

            root.after(0, update_gui_with_speech)
        else:
            def update_gui_no_speech():
                output_text.delete("1.0", tk.END)
                output_text.insert(tk.END, "Sorry, no speech detected.")
                status_label.config(text="Ready", fg="green")

            root.after(0, update_gui_no_speech)

    threading.Thread(target=background_speech_recognition, daemon=True).start()

# New function to show preferred languages in a popup
def show_languages():
    languages_list = "\n".join([f"{code}: {name}" for code, name in LANGUAGES.items()])
    messagebox.showinfo("Available Preferred Languages", languages_list)

# GUI setup
root = tk.Tk()
root.title("Language Detection and Translation App")
root.geometry("650x650")  # Set the window size

# Styling for modern appearance
root.config(bg="#f1f1f1")

# Create frames to organize widgets
header_frame = tk.Frame(root, bg="#f1f1f1", pady=10)
header_frame.pack(fill="x")

input_frame = tk.Frame(root, bg="#f1f1f1", padx=10, pady=10)
input_frame.pack(fill="x", pady=10)

output_frame = tk.Frame(root, bg="#f1f1f1", padx=10, pady=10)
output_frame.pack(fill="x", pady=10)

language_frame = tk.Frame(root, bg="#f1f1f1", padx=10, pady=10)
language_frame.pack(fill="x", pady=10)

# Create labels and input fields
language_code_label = tk.Label(input_frame, text="Select Preferred Language:", font=('Helvetica', 10, 'bold'), bg="#f1f1f1")
language_code_label.grid(row=0, column=0, sticky="w")

preferred_language_var = tk.StringVar(value='en')  # Default value is 'en'
preferred_language_menu = tk.OptionMenu(input_frame, preferred_language_var, *LANGUAGES.keys())
preferred_language_menu.grid(row=0, column=1, pady=5, sticky="ew")

text_label = tk.Label(input_frame, text="Enter Text to Detect Language or Convert to Speech:", font=('Helvetica', 10, 'bold'), bg="#f1f1f1")
text_label.grid(row=1, column=0, columnspan=2, pady=10)

text_entry = tk.Text(input_frame, height=4, width=50, font=('Helvetica', 12))
text_entry.grid(row=2, column=0, columnspan=2, pady=5)

status_label = tk.Label(language_frame, text="Ready", font=('Helvetica', 12, 'bold'), fg="green", bg="#f1f1f1")
status_label.grid(row=0, column=0, pady=5)

output_label = tk.Label(output_frame, text="Recognized Speech Output:", font=('Helvetica', 10, 'bold'), bg="#f1f1f1")
output_label.grid(row=0, column=0, sticky="w")

output_text = tk.Text(output_frame, height=4, width=50, font=('Helvetica', 12))
output_text.grid(row=1, column=0, pady=5)

submit_button = tk.Button(input_frame, text="Submit Text", command=submit_text, font=('Helvetica', 12, 'bold'), bg="#4CAF50", fg="white", padx=15, pady=8, relief="raised")
submit_button.grid(row=3, column=0, pady=5, sticky="ew")

speech_button = tk.Button(input_frame, text="Convert Text to Speech", command=convert_to_speech, font=('Helvetica', 12, 'bold'), bg="#FF9800", fg="white", padx=15, pady=8, relief="raised")
speech_button.grid(row=4, column=0, pady=5, sticky="ew")

# Add the speech recognition button
speech_recognition_button = tk.Button(input_frame, text="Speech Recognition", command=handle_speech_gui, font=('Helvetica', 12, 'bold'), bg="#2196F3", fg="white", padx=15, pady=8, relief="raised")
speech_recognition_button.grid(row=5, column=0, pady=5, sticky="ew")

# Add the button to show available preferred languages
language_info_button = tk.Button(input_frame, text="Show Available Languages", command=show_languages, font=('Helvetica', 12, 'bold'), bg="#9C27B0", fg="white", padx=15, pady=8, relief="raised")
language_info_button.grid(row=6, column=0, pady=5, sticky="ew")

exit_button = tk.Button(input_frame, text="Exit", command=root.quit, font=('Helvetica', 12, 'bold'), bg="#f44336", fg="white", padx=15, pady=8, relief="raised")
exit_button.grid(row=7, column=0, pady=20, sticky="ew")

# Label to show the detected language
detected_language_label = tk.Label(language_frame, text="Detected Language: ", font=('Helvetica', 10, 'italic'), bg="#f1f1f1")
detected_language_label.grid(row=1, column=0, pady=10)

# Label to show the preferred language
preferred_language_label = tk.Label(language_frame, text="Preferred Language: ", font=('Helvetica', 10, 'italic'), bg="#f1f1f1")
preferred_language_label.grid(row=2, column=0, pady=10)

root.mainloop()